# 🤖 AI Engineering Fundamentals — Lezione 1
## Notebook Gruppo A

**ITS Novitas 4.0 | Martedì 19/05/2026**

---

### 📋 Istruzioni
1. Cliccate **File → Salva una copia in Drive** prima di iniziare
2. Configurate la API key nei Secrets (🔑 nella barra sinistra)
3. Lavorate in gruppo — discutete le risposte prima di scrivere
4. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo
Scrivete i vostri nomi qui sotto: Alfonso, Carlo, Lorenzo.

In [9]:
GRUPPO = "A"
MEMBRI = [
    "Alfonso",  # ← Nome 1
    "Carlo",  # ← Nome 2
    "Lorenzo",  # ← Nome 3
    "",  # ← Nome 4 (se presente)
]

print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo A — Alfonso, Carlo, Lorenzo


In [10]:
# Setup — eseguite questa cella per prima
!pip install anthropic -q

from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(messaggio, temperature=0.7, system=None, max_tokens=500):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

✅ Setup completato!


---
## 🎯 Tema del Gruppo A: Token e Costi

Il vostro gruppo esplora come il testo viene trasformato in token
e come questo impatta i costi delle API.

---
### Esercizio 1 — Esplorare i token *(guidato)*

Ogni chiamata all'API restituisce anche il numero di token usati.
Completate il codice qui sotto per stampare input e output token.

In [11]:
# Esercizio 1 — completate le parti mancanti

risposta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[{"role": "user", "content": "Cos'è l'intelligenza artificiale? Rispondi in 2 frasi."}]
)

# TODO: stampate il testo della risposta
print("Risposta:", risposta.content[0].text)

# TODO: stampate il numero di token in input
print("Token input:", risposta.usage.input_tokens)

# TODO: stampate il numero di token in output
print("Token output:", risposta.usage.output_tokens)

# TODO: calcolate il costo stimato
# Haiku: $1 per milione di token input, $5 per milione di token output
costo = (risposta.usage.input_tokens / 1_000_000 * 1.0) + (risposta.usage.output_tokens / 1_000_000 * 5.0)
print(f"Costo stimato: ${costo:.6f}")

Risposta: L'intelligenza artificiale è un insieme di tecnologie e algoritmi che permettono ai computer di eseguire compiti che normalmente richiedono intelligenza umana, come imparare dai dati, riconoscere modelli e prendere decisioni. Viene utilizzata in moltissimi ambiti: dal riconoscimento vocale ai sistemi di raccomandazione, dalle auto autonome all'analisi medica.
Token input: 29
Token output: 100
Costo stimato: $0.000529


---
### Esercizio 2 — Italiano vs Inglese *(guidato)*

La stessa frase in italiano consuma più token che in inglese.
Verificatelo empiricamente.

In [12]:
# Esercizio 2 — confronto token italiano vs inglese

frase_it = "L'intelligenza artificiale sta cambiando il mondo del lavoro in modo profondo."
frase_en = "Artificial intelligence is deeply changing the world of work."

def conta_token(testo):
    """Conta i token di un testo senza inviarlo al modello."""
    result = client.messages.count_tokens(
        model="claude-haiku-4-5-20251001",
        messages=[{"role": "user", "content": testo}]
    )
    return result.input_tokens

token_it = conta_token(frase_it)
token_en = conta_token(frase_en)

print(f"Frase IT: '{frase_it}'")
print(f"Token IT: {token_it}")
print()
print(f"Frase EN: '{frase_en}'")
print(f"Token EN: {token_en}")
print()
print(f"Differenza: {token_it - token_en} token in più per l'italiano")
print(f"Overhead: +{((token_it/token_en)-1)*100:.0f}%")

# Discussione: il risultato vi sorprende? Perché secondo voi succede?
# Scrivete la vostra risposta come commento qui sotto:
# Succede per una differenza grammaticale tra italiano e inglese, dove l'italiano risulta essere più prolisso. La cosa non ci sorprende!

Frase IT: 'L'intelligenza artificiale sta cambiando il mondo del lavoro in modo profondo.'
Token IT: 27

Frase EN: 'Artificial intelligence is deeply changing the world of work.'
Token EN: 18

Differenza: 9 token in più per l'italiano
Overhead: +50%


---
### Esercizio 3 — Quanto costa una conversazione? *(libero)*

Simulate una conversazione di **5 domande e risposte** sul tema
dei prodotti WiData. Alla fine calcolate il costo totale.

**Domande da fare:**
1. Cosa fa WiData?
2. Che tipo di sensori offrite?
3. Funzionano all'aperto?
4. Come si connettono al cloud?
5. Quanto costano?

*(Non preoccupatevi se le risposte non sono accurate — Claude non conosce WiData. Lo correggeremo con RAG nella Lezione 4.)*

In [15]:
# Esercizio 3 — simulate la conversazione e calcolate il costo totale
# Questo esercizio è libero: scrivete voi il codice

domande = [
    "Cosa fa WiData?",
    "Che tipo di sensori offrite?",
    "Funzionano all'aperto?",
    "Come si connettono al cloud?",
    "Quanto costano?",
]

# TODO: per ogni domanda:
# 1. chiamate l'API
# 2. stampate domanda e risposta
# 3. accumulate i token usati

token_input_totale = 0
token_output_totale = 0

for domanda in domande:
  token_input_totale += conta_token(domanda)
  risposta = chiedi_claude(domanda)
  token_output_totale += conta_token(risposta)
  print("Domanda: ", domanda)
  print("=" *20)
  print("Risposta: ", risposta)

# ...

# TODO: alla fine calcolate e stampate il costo totale
# ...

token_conversazione = token_input_totale + token_output_totale
print("Token totale conversazione: ", token_conversazione)

costo_conversazione = token_conversazione / 1_000_000 * 5.0
print(f"Costo totale della conversazione: ${costo_conversazione}")



Domanda:  Cosa fa WiData?
Risposta:  # WiData

WiData è un'azienda italiana specializzata in **data analytics e business intelligence**. Le sue principali attività includono:

## Servizi principali
- **Analisi dei dati** - Elaborazione e interpretazione di grandi volumi di dati
- **Business Intelligence** - Creazione di dashboard e report per supportare le decisioni aziendali
- **Data Strategy** - Consulenza per definire strategie di utilizzo dei dati
- **Implementazione di soluzioni** - Deployment di piattaforme di analytics

## Settori di riferimento
Lavora principalmente con aziende che necessitano di:
- Trasformazione digitale
- Ottimizzazione dei processi decisionali
- Valorizzazione del patrimonio dati

Se stai cercando informazioni più specifiche su WiData, puoi dirmi:
- Se hai una domanda su un servizio particolare
- Se ti interessa conoscere i loro clienti o case study
- Se vuoi sapere come contattarli

Hai altre domande su questa azienda?
Domanda:  Che tipo di sensori offrite

---
### Esercizio 4 — Ottimizzare i token *(libero)*

Il system prompt occupa token ad ogni chiamata.
Confrontate due versioni dello stesso system prompt:
- **Versione lunga**: almeno 100 parole
- **Versione corta**: massimo 20 parole, stesso significato

Quanti token risparmiate? E la qualità delle risposte cambia?

In [18]:
# Esercizio 4 — confronto system prompt lungo vs corto

system_lungo = """
Sei Aria, l'assistente virtuale di Widata, una piattaforma dedicata a servizi per smart cities. Il tuo ruolo è aiutare gli utenti in modo cordiale, professionale e preciso, offrendo risposte chiare e soluzioni concrete.
Identità e tono Comunichi in italiano con uno stile cortese ma diretto, evitando formalismi eccessivi. Dai del "tu" all'utente, salvo passare al "lei" se l'utente si rivolge a te con un registro più formale.
Mantieni un tono positivo e disponibile, senza risultare artificioso o eccessivamente entusiasta. Rispondi con grande dettaglio

"""

system_corto = """
Sei l'assistente virutale di Widata. Rispondi nel modo più breve possibile
"""

domanda_test = "Quali prodotti offrite?"

# TODO: chiamate l'API con entrambi i system prompt
# Confrontate: token usati, costo, qualità della risposta

risposta_corta = chiedi_claude(domanda, system=system_corto)
token_corta = conta_token(risposta_corta)

risposta_lunga = chiedi_claude(domanda, system=system_lungo)
token_lunga = conta_token(risposta_lunga)

#Domanda
print("Domanda: ", domanda_test)
#Risposte
print(f"""\n Risposta corta:
          {risposta_corta}""")
print(f"""\n Risposta lunga:
          {risposta_lunga}""")
print(f"\n I token della risposta corta sono: {token_corta}, i token della risposta lunga sono: {token_lunga}")

#Costi
costo_corta = token_corta / 1_000_000 * 5.0
costo_lunga = token_lunga / 1_000_000 * 5.0
print(f"\n Il costo della chiamata corta era : ${costo_corta:.6f}, il costo della chiamata lunga era: ${costo_lunga:.6f}")

# Conclusione del gruppo (scrivete come commento):
# Vale la pena ottimizzare il system prompt? Quando sì, quando no?
# Vale sicuramente la pena di ottimizzare il system prompt per ottenere risposte più aderenti alle nostre esigenze, mantenendo costi ridotti.

Domanda:  Quali prodotti offrite?

 Risposta corta:
          Non ho abbastanza contesto per rispondere. Puoi specificare a cosa ti riferisci? 

Ad esempio:
- Quali prodotti/servizi Widata?
- Quale piano o soluzione?

Così potrò aiutarti meglio! 😊

 Risposta lunga:
          Ciao! 👋

Apprezzo la domanda, ma per darti un'informazione precisa sui costi, mi servirebbero alcuni dettagli in più:

**Cosa stai cercando?**
- Sei interessato a servizi specifici per la tua città o comune?
- Hai un progetto smart city in mente?
- Vuoi conoscere i prezzi di una soluzione particolare?

**Widata offre soluzioni diverse**, che possono includere:
- **Piattaforme di gestione dati urbani**
- **Servizi di analisi e monitoraggio**
- **Soluzioni IoT e sensori**
- **Consulenza e implementazione**

I costi variano molto in base a:
- La dimensione della città/area
- Il numero di servizi attivati
- La complessità dell'implementazione
- Il periodo di contratto

**Ti suggerisco di:**
1. Dirmi quali servizi ti in

---
## 📊 Preparate la presentazione

Avete **30 minuti** per completare gli esercizi e preparare **5 slide**.

Le slide devono rispondere a:
1. **Cos'è un token?** (con un esempio concreto che avete scoperto)
2. **Perché l'italiano costa di più?** (con i numeri che avete misurato)
3. **Quanto costa una conversazione reale?** (con i vostri risultati)
4. **Vale la pena ottimizzare il system prompt?** (con la vostra conclusione)
5. **La cosa più sorprendente** che avete scoperto

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*